# Real fungicide field trial: hop downy mildew

This notebook executes the public real-data case in v0.2.0. It focuses on experimental structure,
a source-defined confounded year, matched timing contrasts, uncertainty and prediction into an
unseen trial year.

Source paper: Richardson & Gent (2024), *Plant Health Progress*,
DOI `10.1094/PHP-10-23-0086-BR`.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from crop_protection_ps.hop_trial import (
    bootstrap_timing_effects,
    compare_year_transportability,
    load_hop_trial,
    paired_timing_differences,
    prepare_primary_analysis,
    prepare_timing_analysis,
    treatment_timing_summary,
    year_severity_table,
)
from crop_protection_ps.real_demo import run_real_demo


## 1. Load the public experiment

Period symbols in the source CSV are parsed as missing values. The source investigators excluded
2019 because flooding confounded disease measurement; the raw year remains in the data and the
exclusion is applied only in the primary-analysis function.


In [2]:
raw_path = ROOT / "data" / "raw" / "richardson_gent_hop_downy_mildew.csv"
raw = load_hop_trial(raw_path)
primary = prepare_primary_analysis(raw)
timing = prepare_timing_analysis(primary)

print(f"Raw rows: {len(raw)}")
print(f"Missing AUDPC: {raw.audpc.isna().sum()}")
print(f"Primary rows: {len(primary)}")
print(f"Timing-analysis rows: {len(timing)}")
year_severity_table(raw)


Raw rows: 290
Missing AUDPC: 3
Primary rows: 227
Timing-analysis rows: 187


,year,all_treatments_mean_audpc,n_observed,untreated_mean_audpc,untreated_sd_audpc,primary_analysis
0,2017,192.053617,47,243.386,125.267254,True
1,2018,125.903333,60,188.048,62.662610,True
2,2019,58.424667,60,46.764,24.896521,False
3,2020,680.544000,60,909.146,157.812158,True
4,2021,477.944833,60,941.734,151.334463,True


## 2. Match Early and Late programmes within the experimental structure

The estimand is formed within the same **year × block × product**. Positive `Late − Early` AUDPC
means greater disease under the Late programme.


In [3]:
paired = paired_timing_differences(timing)
effects = bootstrap_timing_effects(paired, n_bootstrap=5_000, seed=20260826)
print(f"Complete matched pairs: {len(paired)}")
effects


Complete matched pairs: 92


,scope,treatment,n_pairs,mean_delta_audpc_late_minus_early,ci95_low_audpc,ci95_high_audpc,mean_delta_sqrt_audpc_late_minus_early,ci95_low_sqrt_audpc,ci95_high_sqrt_audpc
0,all_products,ALL,92,40.931522,15.542572,67.207237,1.251892,0.512856,1.996056
1,product,Curzate,18,46.392222,11.796917,79.370389,1.610773,0.644355,2.510087
2,product,FungiPhi,19,21.576316,-33.386803,75.414645,0.919401,-0.562743,2.358516
3,product,Presidio,15,69.843333,-21.152733,156.431533,2.086631,-0.304424,4.662682
4,product,Ranman,20,56.553000,0.807875,119.819100,1.256186,-0.194829,2.823606
5,product,Revus,20,17.099000,-15.837150,50.969387,0.614419,-0.414961,1.660286


The aggregate interval resamples whole year-block clusters,
preserving dependence between product contrasts observed in the same experimental block.
Product-specific intervals are retained because the timing effect is heterogeneous.


In [4]:
treatment_timing_summary(timing).round(3)

,treatment,timing,n,mean_audpc,sd_audpc,mean_control_efficacy,sd_control_efficacy
0,Curzate,Early,20,291.834,227.262,0.518,0.207
1,Curzate,Late,18,356.366,235.569,0.387,0.292
2,FungiPhi,Early,20,361.588,246.508,0.358,0.207
3,FungiPhi,Late,19,397.411,218.387,0.223,0.289
4,Presidio,Early,15,382.435,267.887,0.467,0.213
5,Presidio,Late,15,452.278,262.536,0.204,0.543
6,Ranman,Early,20,338.588,230.398,0.312,0.326
7,Ranman,Late,20,395.141,259.800,0.281,0.235
8,Revus,Early,20,311.400,238.423,0.446,0.220
9,Revus,Late,20,328.499,225.110,0.417,0.190


## 3. Transportability audit

A categorical trial-year effect can be useful for adjustment when the same years are represented
in train and test folds, but it cannot describe a genuinely future year. The benchmark below
therefore compares random CV with leave-one-year-out validation.


In [5]:
transport = compare_year_transportability(timing, seed=20260826)
transport.round(4)


,target,strategy,features,n_folds,rmse,mae,r2,loyo_rmse_increase_pct
0,sqrt_audpc,random_5fold,treatment + timing + year,5,2.7165,2.1763,0.8319,210.2043
1,sqrt_audpc,leave_one_year_out,treatment + timing + year,4,8.4268,7.4980,-0.6175,210.2043
2,relative_disease,random_5fold,treatment + timing,5,0.2853,0.2096,0.0362,9.3697
3,relative_disease,leave_one_year_out,treatment + timing,4,0.3120,0.2404,-0.1528,9.3697


For raw `sqrt(AUDPC)`, random CV is strongly optimistic
because trial year absorbs background disease pressure. Normalising to the contemporaneous
untreated control and removing year from the feature set reduces the random-to-unseen-year RMSE
gap, but the remaining low predictive R² shows that treatment and timing alone are not enough.
A production model needs measured environmental and biological drivers.


## 4. Execute and persist the full real-data case

In [6]:
summary = run_real_demo(ROOT)
summary


{'source': {'url': 'https://github.com/DavidGent-Lab/Richardon-and-Gent-2024-Plant-Health-Progress/blob/main/Data%20Set.csv',
  'local_file': 'data/raw/richardson_gent_hop_downy_mildew.csv',
  'local_sha256': '4056c7290d264f5e9dda0f509dfd74cffdbf816f05de24e967a6c9f72b2768db'},
 'data': {'raw_rows': 290,
  'missing_audpc': 3,
  'years': [2017, 2018, 2019, 2020, 2021],
  'source_excluded_year': 2019,
  'primary_rows': 227,
  'timing_analysis_rows': 187,
  'paired_timing_comparisons': 92},
 'paired_timing': {'interpretation': 'positive late-minus-early means greater disease under late timing',
  'mean_delta_audpc_late_minus_early': 40.93152173913044,
  'ci95_audpc': [15.542572271045325, 67.2072367021276],
  'mean_delta_sqrt_audpc_late_minus_early': 1.2518921338241278,
  'ci95_sqrt_audpc': [0.5128564942506636, 1.9960563848270307]},
 'transportability': {'raw_sqrt_audpc_random_rmse': 2.7165449283787946,
  'raw_sqrt_audpc_leave_one_year_out_rmse': 8.426839372667862,
  'raw_rmse_increase_pct'